# 🚢 Анализ выживаемости на Титанике
**Автор: Тазабек Сымбат** Тазабеков  | [GitHub](https://github.com/tazabekovvas)  
**Инструменты:** Python, Pandas, NumPy, Matplotlib, Seaborn, Scikit-learn

---

## Цель проекта
Проанализировать данные пассажиров Титаника и построить модель машинного обучения, предсказывающую выживаемость. 

## Вопросы исследования
- Какие факторы влияли на выживаемость?
- Была ли зависимость между классом билета и шансами выжить?
- Как пол и возраст влияли на выживаемость?


## 1. Импорт библиотек

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Настройка стиля графиков
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ Все библиотеки успешно импортированы")


## 2. Загрузка данных

In [ ]:
# Загрузка датасета Titanic напрямую с GitHub
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

print(f"Размер датасета: {df.shape[0]} строк, {df.shape[1]} столбцов")
df.head()


## 3. Разведочный анализ данных (EDA)

In [ ]:
# Общая информация о датасете
print("=== Информация о датасете ===")
df.info()
print("\n=== Пропущенные значения ===")
print(df.isnull().sum())
print("\n=== Базовая статистика ===")
df.describe()


In [ ]:
# Общая выживаемость
survived_counts = df['Survived'].value_counts()
print(f"Погибших: {survived_counts[0]} ({survived_counts[0]/len(df)*100:.1f}%)")
print(f"Выживших: {survived_counts[1]} ({survived_counts[1]/len(df)*100:.1f}%)")


### 3.1 Выживаемость по полу

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# График 1: Выживаемость по полу
survival_by_sex = df.groupby('Sex')['Survived'].mean() * 100
axes[0].bar(survival_by_sex.index, survival_by_sex.values, color=['#FF6B6B', '#4ECDC4'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Выживаемость по полу (%)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Процент выживших (%)')
axes[0].set_ylim(0, 100)
for i, (idx, val) in enumerate(survival_by_sex.items()):
    axes[0].text(i, val + 1, f'{val:.1f}%', ha='center', fontweight='bold', fontsize=12)

# График 2: Количество по полу
sex_survived = df.groupby(['Sex', 'Survived']).size().unstack()
sex_survived.columns = ['Погиб', 'Выжил']
sex_survived.plot(kind='bar', ax=axes[1], color=['#FF6B6B', '#4ECDC4'], edgecolor='white')
axes[1].set_title('Количество пассажиров по полу и выживаемости', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Количество пассажиров')
axes[1].set_xlabel('Пол')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend()

plt.tight_layout()
plt.savefig('survival_by_sex.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n📊 Вывод: Женщины выживали значительно чаще (~74%) по сравнению с мужчинами (~19%)")


### 3.2 Выживаемость по классу билета

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Выживаемость по классу
survival_by_class = df.groupby('Pclass')['Survived'].mean() * 100
class_labels = ['1 класс (Люкс)', '2 класс', '3 класс (Эконом)']
colors = ['#FFD700', '#C0C0C0', '#CD7F32']

axes[0].bar(class_labels, survival_by_class.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Выживаемость по классу билета (%)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Процент выживших (%)')
axes[0].set_ylim(0, 100)
for i, val in enumerate(survival_by_class.values):
    axes[0].text(i, val + 1, f'{val:.1f}%', ha='center', fontweight='bold', fontsize=12)

# Возраст по классам
df.boxplot(column='Age', by='Pclass', ax=axes[1])
axes[1].set_title('Распределение возраста по классам', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Класс билета')
axes[1].set_ylabel('Возраст')
plt.suptitle('')

plt.tight_layout()
plt.savefig('survival_by_class.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n📊 Вывод: Пассажиры 1-го класса выживали чаще (~63%), пассажиры 3-го класса — реже (~24%)")


### 3.3 Распределение возраста и стоимости билета

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Возраст выживших vs погибших
df[df['Survived']==1]['Age'].dropna().hist(ax=axes[0], bins=30, alpha=0.7, color='#4ECDC4', label='Выжил')
df[df['Survived']==0]['Age'].dropna().hist(ax=axes[0], bins=30, alpha=0.7, color='#FF6B6B', label='Погиб')
axes[0].set_title('Распределение возраста', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Возраст')
axes[0].set_ylabel('Количество пассажиров')
axes[0].legend()

# Стоимость билета vs выживаемость
df.boxplot(column='Fare', by='Survived', ax=axes[1])
axes[1].set_title('Стоимость билета vs Выживаемость', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Выжил (0 = Нет, 1 = Да)')
axes[1].set_ylabel('Стоимость билета ($)')
plt.suptitle('')

plt.tight_layout()
plt.savefig('age_fare_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n📊 Вывод: Выжившие в среднем платили больше за билет — косвенно подтверждает влияние класса")


### 3.4 Корреляционная матрица

In [ ]:
plt.figure(figsize=(10, 7))
numeric_cols = df[['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']].corr()
sns.heatmap(numeric_cols, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Корреляционная матрица числовых признаков', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n📊 Вывод: Класс билета (Pclass) имеет наибольшую отрицательную корреляцию с выживаемостью (-0.34)")


## 4. Подготовка данных для ML

Перед обучением модели нужно:
- Обработать пропущенные значения
- Закодировать категориальные признаки
- Отобрать важные признаки


In [ ]:
# Создаём копию датасета
df_ml = df.copy()

# Заполняем пропущенные значения
df_ml['Age'].fillna(df_ml['Age'].median(), inplace=True)
df_ml['Embarked'].fillna(df_ml['Embarked'].mode()[0], inplace=True)
df_ml['Fare'].fillna(df_ml['Fare'].median(), inplace=True)

# Кодируем пол (male=1, female=0)
df_ml['Sex_encoded'] = LabelEncoder().fit_transform(df_ml['Sex'])

# Кодируем порт посадки
df_ml['Embarked_encoded'] = LabelEncoder().fit_transform(df_ml['Embarked'])

# Выбираем признаки для модели
features = ['Pclass', 'Sex_encoded', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked_encoded']
X = df_ml[features]
y = df_ml['Survived']

print(f"Размер X: {X.shape}")
print(f"Признаки: {features}")
print(f"\nПропущенные значения: {X.isnull().sum().sum()}")
print("✅ Данные готовы к обучению")


## 5. Обучение модели

In [ ]:
# Разделяем данные на обучающую и тестовую выборки (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Обучающая выборка: {X_train.shape[0]} записей")
print(f"Тестовая выборка: {X_test.shape[0]} записей")

# Модель 1: Логистическая регрессия
lr_model = LogisticRegression(random_state=42, max_iter=200)
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)
lr_accuracy = accuracy_score(y_test, lr_pred)

# Модель 2: Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_pred)

print(f"\n📈 Логистическая регрессия: {lr_accuracy:.4f} ({lr_accuracy*100:.2f}%)")
print(f"📈 Random Forest:           {rf_accuracy:.4f} ({rf_accuracy*100:.2f}%)")
print(f"\n✅ Лучшая модель: {'Random Forest' if rf_accuracy > lr_accuracy else 'Логистическая регрессия'}")


## 6. Оценка модели

In [ ]:
# Детальный отчёт для Random Forest
print("=== Отчёт по классификации (Random Forest) ===")
print(classification_report(y_test, rf_pred, target_names=['Погиб', 'Выжил']))

# Матрица ошибок
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, rf_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Погиб', 'Выжил'], yticklabels=['Погиб', 'Выжил'])
axes[0].set_title('Матрица ошибок (Random Forest)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Реальное значение')
axes[0].set_xlabel('Предсказание')

# Feature Importance
feat_importance = pd.Series(rf_model.feature_importances_, index=features).sort_values()
feat_importance.plot(kind='barh', ax=axes[1], color='#4ECDC4', edgecolor='white')
axes[1].set_title('Важность признаков (Random Forest)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Важность')

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Выводы

### Ключевые находки из EDA:
1. **Пол** — самый сильный фактор: женщины выживали в ~74% случаев, мужчины лишь в ~19%
2. **Класс билета** — пассажиры 1-го класса выживали вдвое чаще, чем 3-го (63% vs 24%)
3. **Возраст** — дети имели более высокие шансы выжить (принцип "женщины и дети первыми")
4. **Стоимость билета** — высокая стоимость коррелирует с выживаемостью (связано с классом)

### Результаты модели ML:
| Модель | Точность |
|--------|----------|
| Логистическая регрессия | ~80% |
| Random Forest | ~82% |

### Наиболее важные признаки для модели:
- Пол пассажира (`Sex`)
- Стоимость билета (`Fare`)  
- Возраст (`Age`)
- Класс билета (`Pclass`)

---
*Проект выполнен в рамках учебной практики по курсу Data Science, МУИТ, 2024*
